# sklearn / TensorFlow — Course 3 (Unsupervised Learning, Recommenders, Reinforcement Learning)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets      import make_blobs, make_classification
from sklearn.pipeline      import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster       import KMeans
from sklearn.covariance    import EllipticEnvelope
from sklearn.ensemble      import IsolationForest
from sklearn.decomposition import PCA
from sklearn.metrics       import f1_score

import tensorflow as tf
from tensorflow.keras            import Sequential, Model
from tensorflow.keras.layers     import Dense, Input, Dot
from tensorflow.keras.optimizers import Adam

from collections import deque
import random

np.random.seed(42)
tf.random.set_seed(42)   # reproducible results, same idea as random_state in sklearn

---
# PART A — K-MEANS CLUSTERING
Unsupervised — group unlabeled points into K clusters based on distance to centroids.

## 2a. Generate Data


In [ ]:
X_cluster, true_labels = make_blobs(
    n_samples=300, centers=4, cluster_std=1.2, random_state=42
)

plt.scatter(X_cluster[:, 0], X_cluster[:, 1], alpha=0.6)
plt.title('Raw Unlabeled Data')
plt.show()

## 3a. Elbow Method — Choose K
Plot inertia (distortion J) against K and look for where the drop levels off.

In [ ]:
inertias = []
K_range = range(1, 11)

for k in K_range:
    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('kmeans', KMeans(n_clusters=k, n_init=50, random_state=42)),
    ])
    pipe.fit(X_cluster)
    inertias.append(pipe.named_steps['kmeans'].inertia_)

plt.plot(K_range, inertias, marker='o')
plt.xlabel('K')
plt.ylabel('Inertia (Distortion J)')
plt.title('Elbow Method')
plt.show()

## 4a. Fit Final K-Means Model
Feature scaling matters here — K-means uses distance, so unscaled features would dominate.

In [ ]:
K_FINAL = 4

kmeans_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('kmeans', KMeans(n_clusters=K_FINAL, n_init=50, random_state=42)),
])
kmeans_pipe.fit(X_cluster)

cluster_labels = kmeans_pipe.predict(X_cluster)
centroids_scaled = kmeans_pipe.named_steps['kmeans'].cluster_centers_

print(f'Final inertia (J): {kmeans_pipe.named_steps["kmeans"].inertia_:.2f}')

## 5a. Visualize Clusters

In [ ]:
X_scaled_view = kmeans_pipe.named_steps['scale'].transform(X_cluster)

plt.scatter(X_scaled_view[:, 0], X_scaled_view[:, 1], c=cluster_labels, cmap='viridis', alpha=0.6)
plt.scatter(centroids_scaled[:, 0], centroids_scaled[:, 1], c='red', marker='X', s=200, label='centroids')
plt.legend()
plt.title('K-Means Clustering Result')
plt.show()

## 6a. Distortion Cost — Hand-Rolled (matches course formula)
`J = (1/m) * sum(||x[i] - mu[c[i]]||^2)` — should match `inertia_` above (up to the 1/m normalization).

In [ ]:
def compute_distortion(X, centroids, labels):
    m = X.shape[0]
    total = 0.0
    for i in range(m):
        total += np.sum((X[i] - centroids[labels[i]]) ** 2)
    return total / m

J = compute_distortion(X_scaled_view, centroids_scaled, cluster_labels)
print(f'Hand-rolled J (mean squared distance): {J:.4f}')

---
# PART B — ANOMALY DETECTION
Fit a Gaussian to mostly-normal data, then flag new points with low probability p(x) as anomalies.

## 2b. Generate Data — Normal Examples + A Few Injected Anomalies

In [ ]:
X_normal, _ = make_blobs(n_samples=300, centers=1, cluster_std=1.0, random_state=1)
X_anomalies = np.random.uniform(low=-8, high=8, size=(15, 2))

X_all = np.vstack([X_normal, X_anomalies])
y_all = np.array([0] * len(X_normal) + [1] * len(X_anomalies))   # 1 = true anomaly, for evaluation only

X_train_norm = X_normal[:250]                 # train Gaussian on normal data ONLY
X_val_anom, y_val_anom = X_all[250:], y_all[250:]

plt.scatter(X_normal[:, 0], X_normal[:, 1], alpha=0.5, label='normal')
plt.scatter(X_anomalies[:, 0], X_anomalies[:, 1], color='red', marker='x', label='injected anomaly')
plt.legend()
plt.title('Data with Injected Anomalies')
plt.show()

## 3b. Fit Gaussian Parameters (mu, sigma^2) — Hand-Rolled
Matches the course's per-feature independent Gaussian formula.

In [ ]:
def estimate_gaussian(X):
    mu = np.mean(X, axis=0)
    var = np.var(X, axis=0)
    return mu, var

def gaussian_probability(X, mu, var):
    # p(x) = product over features of the 1D Gaussian pdf
    p = np.ones(X.shape[0])
    for j in range(X.shape[1]):
        coeff = 1 / np.sqrt(2 * np.pi * var[j])
        exponent = -((X[:, j] - mu[j]) ** 2) / (2 * var[j])
        p *= coeff * np.exp(exponent)
    return p

mu, var = estimate_gaussian(X_train_norm)
p_val = gaussian_probability(X_val_anom, mu, var)

print(f'mu:  {mu}')
print(f'var: {var}')

## 4b. Select Epsilon Threshold using F1-score
Anomalies are rare (imbalanced) — accuracy is misleading, so tune epsilon against F1 on a small labeled validation split.

In [ ]:
def select_epsilon(p_val, y_val):
    best_eps, best_f1 = 0, 0
    for eps in np.linspace(p_val.min(), p_val.max(), 1000):
        preds = (p_val < eps).astype(int)
        f1 = f1_score(y_val, preds)
        if f1 > best_f1:
            best_f1, best_eps = f1, eps
    return best_eps, best_f1

best_epsilon, best_f1 = select_epsilon(p_val, y_val_anom)
print(f'Best epsilon: {best_epsilon:.6e}')
print(f'Best F1     : {best_f1:.4f}')

## 5b. Flag Anomalies & Visualize

In [ ]:
flagged = p_val < best_epsilon

plt.scatter(X_val_anom[~flagged, 0], X_val_anom[~flagged, 1], alpha=0.5, label='normal')
plt.scatter(X_val_anom[flagged, 0], X_val_anom[flagged, 1], color='red', marker='x', s=100, label='flagged anomaly')
plt.legend()
plt.title('Flagged Anomalies (epsilon threshold)')
plt.show()

## 6b. sklearn Alternative — IsolationForest
Doesn't assume a Gaussian shape, more robust in practice than the hand-rolled version above.

In [ ]:
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_forest.fit(X_train_norm)

preds_iso = iso_forest.predict(X_val_anom)          # 1 = normal, -1 = anomaly
preds_iso_binary = (preds_iso == -1).astype(int)

f1_iso = f1_score(y_val_anom, preds_iso_binary)
print(f'IsolationForest F1: {f1_iso:.4f}')

---
# PART C — RECOMMENDER SYSTEMS: COLLABORATIVE FILTERING
Learn user parameters (w, b) AND movie features (x) simultaneously from a sparse ratings matrix.
No item metadata needed — pure collaborative signal.

## 2c. Create Synthetic Ratings Matrix
`Y[i,j]` = rating user j gave movie i. `R[i,j]` = 1 if that rating exists, else 0 (most entries are unrated).

In [ ]:
n_movies, n_users, n_features_true = 20, 12, 3

# ground-truth latent vectors, used only to GENERATE plausible synthetic ratings
X_true = np.random.randn(n_movies, n_features_true)
W_true = np.random.randn(n_users, n_features_true)
b_true = np.random.randn(1, n_users)

Y_full = X_true @ W_true.T + b_true
Y_full = np.clip(np.round(Y_full + 3), 1, 5)     # squash into a 1-5 star rating range

R = (np.random.rand(n_movies, n_users) < 0.4).astype(int)   # ~40% of ratings observed
Y = Y_full * R

print(f'Ratings matrix shape: {Y.shape}')
print(f'Fraction rated: {R.mean():.2f}')

## 3c. Mean Normalization
Subtract each movie's average rating before training — this way a brand-new user with zero ratings defaults to the movie's average, not to 0.

In [ ]:
def normalize_ratings(Y, R):
    Ymean = np.sum(Y * R, axis=1, keepdims=True) / (np.sum(R, axis=1, keepdims=True) + 1e-12)
    Ynorm = (Y - Ymean) * R
    return Ynorm, Ymean

Ynorm, Ymean = normalize_ratings(Y, R)
print(f'Ymean shape: {Ymean.shape}')

## 4c. Cost Function — Hand-Rolled (matches course notation)
`J(w,b,x) = 0.5 * sum((w.x + b - y)^2) + regularization on w and x`

In [ ]:
def cofi_cost_function(X, W, b, Y, R, lambda_):
    predictions = (X @ W.T) + b
    err = (predictions - Y) * R
    J = 0.5 * np.sum(err ** 2)
    J += (lambda_ / 2) * (np.sum(W ** 2) + np.sum(X ** 2))
    return J

n_features = 10
lambda_ = 1.0

X_init = np.random.randn(n_movies, n_features) * 0.1
W_init = np.random.randn(n_users, n_features) * 0.1
b_init = np.zeros((1, n_users))

initial_cost = cofi_cost_function(X_init, W_init, b_init, Ynorm, R, lambda_)
print(f'Initial cost: {initial_cost:.2f}')

## 5c. Train with TensorFlow GradientTape
Needs custom gradients since we're learning X (movie features) too, not just weights of a fixed network — plain `.fit()` doesn't cover this.

In [ ]:
X_tf = tf.Variable(X_init, dtype=tf.float64)
W_tf = tf.Variable(W_init, dtype=tf.float64)
b_tf = tf.Variable(b_init, dtype=tf.float64)

R_tf = tf.constant(R, dtype=tf.float64)
Ynorm_tf = tf.constant(Ynorm, dtype=tf.float64)

optimizer = Adam(learning_rate=0.1)
cost_history = []

for epoch in range(200):
    with tf.GradientTape() as tape:
        predictions = tf.linalg.matmul(X_tf, W_tf, transpose_b=True) + b_tf
        err = (predictions - Ynorm_tf) * R_tf
        cost = 0.5 * tf.reduce_sum(err ** 2)
        cost += (lambda_ / 2) * (tf.reduce_sum(W_tf ** 2) + tf.reduce_sum(X_tf ** 2))

    grads = tape.gradient(cost, [X_tf, W_tf, b_tf])
    optimizer.apply_gradients(zip(grads, [X_tf, W_tf, b_tf]))
    cost_history.append(cost.numpy())

    if epoch % 20 == 0:
        print(f'epoch {epoch:>4}  cost {cost.numpy():.2f}')

plt.plot(cost_history)
plt.xlabel('Epoch'); plt.ylabel('Cost J')
plt.title('Collaborative Filtering — Training Cost')
plt.show()

## 6c. Make Predictions & Find Similar Movies
Add `Ymean` back after predicting (undo the normalization from step 3c). Similar movies = smallest distance between learned feature vectors.

In [ ]:
predicted_ratings = (X_tf.numpy() @ W_tf.numpy().T + b_tf.numpy()) + Ymean
predicted_ratings = np.clip(predicted_ratings, 1, 5)

print('Predicted ratings for user 0 (first 5 movies):')
print(predicted_ratings[:5, 0])

def find_similar_movies(movie_idx, X_learned, top_n=3):
    distances = np.sum((X_learned - X_learned[movie_idx]) ** 2, axis=1)
    similar_idx = np.argsort(distances)[1:top_n + 1]   # skip index 0 = itself
    return similar_idx

similar = find_similar_movies(0, X_tf.numpy())
print(f'Movies most similar to movie 0: {similar}')

---
# PART D — RECOMMENDER SYSTEMS: CONTENT-BASED FILTERING
Two neural networks (user tower + item tower) each output an embedding of the SAME size.
Prediction = dot product of the two embeddings. Handles cold-start better than collaborative filtering.

## 2d. Synthetic User & Item Features
Unlike Part C, here we actually have known features per user/item (age, genre preference, etc.) instead of learning them from scratch.

In [ ]:
n_samples_cb = 2000
num_user_features = 6
num_item_features = 8

user_features = np.random.randn(n_samples_cb, num_user_features)
item_features = np.random.randn(n_samples_cb, num_item_features)

# synthetic target: some nonlinear-ish combination + noise, squashed to 1-5 star range
true_signal = (user_features[:, 0] * item_features[:, 0]
               + user_features[:, 1] * item_features[:, 1]
               + 0.5 * np.random.randn(n_samples_cb))
y_cb = np.clip(np.round(true_signal + 3), 1, 5)

split = int(0.85 * n_samples_cb)
user_train, user_val = user_features[:split], user_features[split:]
item_train, item_val = item_features[:split], item_features[split:]
y_train_cb, y_val_cb = y_cb[:split], y_cb[split:]

scaler_u = StandardScaler().fit(user_train)
scaler_i = StandardScaler().fit(item_train)
user_train, user_val = scaler_u.transform(user_train), scaler_u.transform(user_val)
item_train, item_val = scaler_i.transform(item_train), scaler_i.transform(item_val)

## 3d. Build Two Towers (User NN + Item NN)
Both output vectors of the same size (`num_outputs`) so their dot product is well-defined. L2-normalizing the embeddings keeps the dot product bounded and training stable.

In [ ]:
num_outputs = 16

user_NN = Sequential([
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(num_outputs),
])

item_NN = Sequential([
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(num_outputs),
])

input_user = Input(shape=(num_user_features,))
vu = user_NN(input_user)
vu = tf.linalg.l2_normalize(vu, axis=1)

input_item = Input(shape=(num_item_features,))
vm = item_NN(input_item)
vm = tf.linalg.l2_normalize(vm, axis=1)

output = Dot(axes=1)([vu, vm])
model_cb = Model([input_user, input_item], output)
model_cb.summary()

## 4d. Compile + Train
Same squared-error idea as collaborative filtering, but now the model just needs standard `.fit()` since the inputs (features) are fixed, not learned.

In [ ]:
model_cb.compile(optimizer=Adam(learning_rate=0.01), loss='mse')

history_cb = model_cb.fit(
    [user_train, item_train], y_train_cb,
    validation_data=([user_val, item_val], y_val_cb),
    epochs=30,
    verbose=0
)
print('done training')

## 5d. Plot Training Loss

In [ ]:
plt.plot(history_cb.history['loss'], label='train loss')
plt.plot(history_cb.history['val_loss'], label='val loss')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Content-Based Filtering — Training vs Validation Loss')
plt.legend()
plt.show()

---
# PART E — PRINCIPAL COMPONENT ANALYSIS (PCA)
Dimensionality reduction, mainly used here for VISUALIZATION — compress high-D data down to 2D to plot it.

## 2e. Generate Higher-Dimensional Data
Reusing a clustering-style dataset, but in more dimensions than we can directly plot.

In [ ]:
X_highdim, labels_highdim = make_blobs(n_samples=300, centers=4, n_features=6, cluster_std=1.5, random_state=42)

X_highdim_scaled = StandardScaler().fit_transform(X_highdim)   # always scale before PCA

pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X_highdim_scaled)

print(f'Explained variance ratio: {pca.explained_variance_ratio_}')
print(f'Total variance captured : {pca.explained_variance_ratio_.sum():.2%}')

## 3e. Visualize in 2D

In [ ]:
plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=labels_highdim, cmap='viridis', alpha=0.7)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('PCA — 6D Data Compressed to 2D')
plt.show()

---
# PART F — REINFORCEMENT LEARNING: DEEP Q-NETWORK (DQN)
Agent learns Q(s,a) via the Bellman equation using a neural network, experience replay, and a slowly-updated target network.
A small custom GridWorld stands in for something like the Lunar Lander environment from the course.

## 2f. Simple Custom Environment — GridWorld
Agent starts at (0,0), goal is at the opposite corner. Actions: 0=up, 1=down, 2=left, 3=right. Reward: -1 per step, +20 at goal, so the agent learns to reach the goal quickly.

In [ ]:
class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.state_dim = 2
        self.n_actions = 4
        self.reset()

    def reset(self):
        self.pos = np.array([0, 0])
        self.goal = np.array([self.size - 1, self.size - 1])
        return self._get_state()

    def _get_state(self):
        # normalize so the network sees small, scaled inputs — same idea as StandardScaler
        return np.concatenate([self.pos, self.goal]) / (self.size - 1)

    def step(self, action):
        moves = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
        dx, dy = moves[action]
        self.pos = np.clip(self.pos + np.array([dx, dy]), 0, self.size - 1)

        done = np.array_equal(self.pos, self.goal)
        reward = 20.0 if done else -1.0
        return self._get_state(), reward, done

env = GridWorld(size=5)
state_dim = env.state_dim * 2   # position + goal, both (x,y)
n_actions = env.n_actions

## 3f. Build Q-Network + Target Network
Two identical networks: the main one trains every step, the target one updates slowly so training targets don't chase a constantly-moving prediction.

In [ ]:
def build_q_network(state_dim, n_actions):
    return Sequential([
        Dense(64, activation='relu'),
        Dense(64, activation='relu'),
        Dense(n_actions),   # linear output — raw Q-values, one per action
    ])

q_network = build_q_network(state_dim, n_actions)
target_q_network = build_q_network(state_dim, n_actions)

# build both networks by calling once, then sync weights
dummy = np.zeros((1, state_dim))
q_network(dummy); target_q_network(dummy)
target_q_network.set_weights(q_network.get_weights())

optimizer_rl = Adam(learning_rate=1e-3)

## 4f. Experience Replay Buffer
Store transitions and train on random mini-batches, instead of training on each step as it happens — breaks harmful correlation between consecutive steps.

In [ ]:
replay_buffer = deque(maxlen=50_000)

def store_experience(s, a, r, s_next, done):
    replay_buffer.append((s, a, r, s_next, done))

def sample_batch(batch_size=64):
    return random.sample(replay_buffer, batch_size)

## 5f. Epsilon-Greedy Action Selection
Start mostly random (explore), decay toward mostly using the learned Q-values (exploit) as training progresses.

In [ ]:
def epsilon_greedy_action(state, epsilon):
    if random.random() < epsilon:
        return random.randint(0, n_actions - 1)
    q_values = q_network(state[np.newaxis, :]).numpy()[0]
    return int(np.argmax(q_values))

## 6f. Training Step — Bellman Target + Gradient Update
`y_target = R(s) + gamma * max_a' Q(s', a')` — this is what the network is trained to predict.

In [ ]:
def soft_update(main_net, target_net, tau=0.01):
    for target_w, main_w in zip(target_net.weights, main_net.weights):
        target_w.assign(tau * main_w + (1 - tau) * target_w)

def train_step(batch, gamma=0.99):
    states, actions, rewards, next_states, dones = zip(*batch)
    states      = tf.convert_to_tensor(states, dtype=tf.float32)
    next_states = tf.convert_to_tensor(next_states, dtype=tf.float32)
    rewards     = tf.convert_to_tensor(rewards, dtype=tf.float32)
    dones       = tf.convert_to_tensor(dones, dtype=tf.float32)
    actions     = tf.convert_to_tensor(actions, dtype=tf.int32)

    max_next_q = tf.reduce_max(target_q_network(next_states), axis=1)
    y_targets  = rewards + (1 - dones) * gamma * max_next_q

    with tf.GradientTape() as tape:
        q_values = q_network(states)
        action_masks = tf.one_hot(actions, depth=n_actions)
        q_selected = tf.reduce_sum(q_values * action_masks, axis=1)
        loss = tf.reduce_mean((q_selected - y_targets) ** 2)

    grads = tape.gradient(loss, q_network.trainable_variables)
    optimizer_rl.apply_gradients(zip(grads, q_network.trainable_variables))
    soft_update(q_network, target_q_network)
    return loss.numpy()

## 7f. Full Training Loop
Run episodes, store experience, train once enough transitions exist, decay epsilon over time.

In [ ]:
EPISODES = 300
BATCH_SIZE = 64
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.98

episode_rewards = []

for ep in range(EPISODES):
    state = env.reset()
    total_reward = 0
    done = False
    steps = 0

    while not done and steps < 50:
        action = epsilon_greedy_action(state, epsilon)
        next_state, reward, done = env.step(action)
        store_experience(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward
        steps += 1

        if len(replay_buffer) >= BATCH_SIZE:
            batch = sample_batch(BATCH_SIZE)
            train_step(batch)

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    episode_rewards.append(total_reward)

    if ep % 50 == 0:
        avg_recent = np.mean(episode_rewards[-50:])
        print(f'episode {ep:>4}  reward {total_reward:>6.1f}  avg(last 50) {avg_recent:>6.1f}  epsilon {epsilon:.3f}')

## 8f. Plot Episode Rewards
Rising trend = the agent is learning to reach the goal in fewer steps (less negative reward accumulated).

In [ ]:
plt.plot(episode_rewards, alpha=0.4, label='per episode')
window = 20
if len(episode_rewards) >= window:
    moving_avg = np.convolve(episode_rewards, np.ones(window) / window, mode='valid')
    plt.plot(range(window - 1, len(episode_rewards)), moving_avg, label=f'{window}-episode moving avg')
plt.xlabel('Episode'); plt.ylabel('Total Reward')
plt.title('DQN Training Progress — GridWorld')
plt.legend()
plt.show()